<a href="https://colab.research.google.com/github/abegithub2024/abegithub2024/blob/main/RF_reg_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Soil Loss Estimation and Classification**


**1)Install and Import All necessary Libraries**

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os

**2) Define pipline**

In [ ]:
def soil_erosion_pipeline(task="regression", data_path="RF_features/soil_erosion_data.csv"):
    """
    Unified pipeline for soil erosion prediction:
    - Regression: continuous soil loss (RUSLE outputs)
    - Classification: binary erosion proxy (NDVI eroded vs non-eroded)
    """

**3)Mount drive to Load training and test data**

In [ ]:
    # Load dataset from Google Drive folder
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Dataset not found at {data_path}")
    df = pd.read_csv(data_path)

    # Feature engineering
    df['rainfall_slope_interaction'] = df['rainfall_intensity'] * df['slope_steepness']
    df['vegetation_protection'] = df['vegetation_cover'] / (df['slope_steepness'] + 1)
    df['rainfall_twi_interaction'] = df['rainfall_intensity'] * df['topographic_wetness_index']
    df['bulk_slope_interaction'] = df['bulk_density'] * df['slope_steepness']
# Feature list (expanded)
    features = [
        'rainfall_intensity','slope_steepness',
        'soil_texture_clay','soil_texture_silt',
        'vegetation_cover','land_use',
        'soil_moisture','organic_matter',
        'rainfall_slope_interaction','vegetation_protection',
        'slope_aspect','topographic_wetness_index',
        'bulk_density','mean_annual_rainfall','mean_annual_temperature',
        'flow_accumulation','stream_power_index','curvature','elevation',
        'soil_erodibility','soil_depth','rock_fragment_content',
        'ndvi_mean','ndvi_variability','crop_type','conservation_practices',
        'rainfall_erosivity','extreme_rainfall_events','wind_speed',
        'road_density','population_density',
        'rainfall_twi_interaction','bulk_slope_interaction'
    ]

In [ ]:
    # Feature list (expanded)
Random Forest Feature Tree
│
├── Rainfall & Climate
│   ├── rainfall_intensity
│   ├── mean_annual_rainfall
│   ├── mean_annual_temperature
│   ├── rainfall_erosivity
│   ├── extreme_rainfall_events
│   ├── wind_speed
│   ├── rainfall_slope_interaction
│   ├── rainfall_twi_interaction
│
├── Topography & Hydrology
│   ├── slope_steepness
│   ├── slope_aspect
│   ├── topographic_wetness_index
│   ├── flow_accumulation
│   ├── stream_power_index
│   ├── curvature
│   ├── elevation
│   ├── bulk_slope_interaction
│
├── Soil Properties
│   ├── soil_texture_clay
│   ├── soil_texture_silt
│   ├── soil_moisture
│   ├── organic_matter
│   ├── bulk_density
│   ├── soil_erodibility
│   ├── soil_depth
│   ├── rock_fragment_content
│
├── Vegetation & Land Use
│   ├── vegetation_cover
│   ├── vegetation_protection
│   ├── ndvi_mean
│   ├── ndvi_variability
│   ├── crop_type
│   ├── land_use
│   ├── conservation_practices
│
├── Human Influence
│   ├── road_density
│   ├── population_density
│

 # Feature list (expanded)
Random Forest Feature Tree
│
├── Rainfall & Climate
│   ├── rainfall_intensity
│   ├── mean_annual_rainfall
│   ├── mean_annual_temperature
│   ├── rainfall_erosivity
│   ├── extreme_rainfall_events
│   ├── wind_speed
│   ├── rainfall_slope_interaction
│   ├── rainfall_twi_interaction
│
├── Topography & Hydrology
│   ├── slope_steepness
│   ├── slope_aspect
│   ├── topographic_wetness_index
│   ├── flow_accumulation
│   ├── stream_power_index
│   ├── curvature
│   ├── elevation
│   ├── bulk_slope_interaction
│
├── Soil Properties
│   ├── soil_texture_clay
│   ├── soil_texture_silt
│   ├── soil_moisture
│   ├── organic_matter
│   ├── bulk_density
│   ├── soil_erodibility
│   ├── soil_depth
│   ├── rock_fragment_content
│
├── Vegetation & Land Use
│   ├── vegetation_cover
│   ├── vegetation_protection
│   ├── ndvi_mean
│   ├── ndvi_variability
│   ├── crop_type
│   ├── land_use
│   ├── conservation_practices
│
├── Human Influence
│   ├── road_density
│   ├── population_density
│

**4)Select Target**

##
**  4.1)Regression**

In [ ]:
       # Select target
    if task == "regression":
        target = 'soil_loss'
        model = RandomForestRegressor(random_state=42, n_jobs=-1)
        scoring = 'r2'
    else:
        target = 'erosion_proxy'
        model = RandomForestClassifier(random_state=42, n_jobs=-1)
        scoring = 'f1'

    X = df[features]
    y = df[target]

    # Preprocessing: scale numeric, one-hot encode categorical
    numeric_features = [f for f in features if f != 'land_use' and f != 'crop_type' and f != 'conservation_practices']
    categorical_features = ['land_use','crop_type','conservation_practices']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ]
    )

    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])

    # Train/test split (replace with spatial CV if watershed groups available)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

**5)Hyperparameter tuning with Randomized Search CV**

In [ ]:
# Hyperparameter tuning with RandomizedSearchCV
    param_dist = {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [5, 10, 15, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    }

    search = RandomizedSearchCV(
        pipeline, param_distributions=param_dist,
        n_iter=10, cv=5, scoring=scoring, n_jobs=-1, random_state=42
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

**6)Prediction**

In [ ]:
    # Predictions
    y_pred = best_model.predict(X_test)

**7) Model Evaluation and Classification of Soil loss hazard zones**

In [ ]:
 # Evaluation
    if task == "regression":
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"Regression Performance:\nRMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
    else:
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1])
        print(f"Classification Performance:\nAcc={acc:.3f}, Prec={prec:.3

Export for visualization

In [ ]:
Export for visualization